[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srujanmp1366/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)


# ML-04  Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order**  each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
print("UNIT OF ANALYSIS:")
print("One row = ONE PAGE (content item), observed over a FIXED 90-DAY TRAILING WINDOW.")
print()
print("GRAIN (what identifies a unique row):")
print("- content_id: pseudonymous page identifier")
print("- Aggregation window: last 90 days of search + analytics data (trailing from export date)")
print()
print("TIME WINDOW:")
print("- Data source: starter dataset `data/raw/content_refresh_anonymized.csv`")
print("- All metrics are pre-aggregated over a 90-day observation window.")
print("- This is SNAPSHOT data, not time-series: one row per page, not one row per date.")
print()
print("WHY THIS UNIT:")
print("- Clustering operates on static page profiles, not temporal sequences.")
print("- Each page's archetype is determined by: what scale it operates at, what quality it delivers, how fresh it is, what type it is.")
print("- All of these are captured in the 90-day window.")
print()
print("IMPLICATION FOR CLUSTERING:")
print("- I'm NOT predicting future decline; I'm identifying observed archetypes from past performance.")
print("- Each cluster profile is a snapshot of 'pages that looked like this over the last 90 days'.")
print("- Actionability: 'Pages in this cluster historically show this pattern  apply this playbook.'")

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
print("DATA CONTRACT: Field Classification for Lane 3 Clustering")
print("="*70)
print()
print("FEATURES (inputs to clustering model):")
print("-" * 70)
print()
print("Numeric (scaled):")
print("  - log_impressions_90d (log1p of impressions  traffic is heavy-tailed)")
print("  - log_clicks_90d (log1p of clicks)")
print("  - log_sessions_90d (log1p of GA4 sessions)")
print("  - ctr (clicks / impressions  100; 100 percentage; 0100 range)")
print("  - avg_position (mean GSC position; 1=best, >50=poor; 0 means 'no data')")
print("  - engagement_rate (engaged_sessions / sessions  100; 0100)")
print("  - scroll_rate (scroll_events / pageviews  100; can exceed 100)")
print("  - content_age_days (days since creation; all rows 90 in this slice)")
print("  - days_since_last_update (freshness)")
print("  - word_count (article length)")
print("  - search_volume (keyword demand)")
print("  - competition (keyword difficulty, 01)")
print()
print("Categorical (encoded):")
print("  - content_type (keyword article / feedly / comparison)")
print("  - main_intent (informational / transactional / commercial / navigational)")
print("  - competition_level (LOW / MEDIUM / HIGH)")
print("  - age_tier (31-90 / 91-180 / 181-365 / 365+)")
print("  - freshness_tier (0-30 / 31-90 / 91-180 / 181+)")
print("  - word_count_tier (<1000 / 1000-2000 / 2000-3500 / 3500+)")
print("  - impression_tier (low / moderate / good / excellent)")
print("  - position_tier (top_3 / page_1 / striking / page_3_5 / deep)")
print()
print()
print("CONTEXT (describe the cluster, not used as features):")
print("-" * 70)
print("  - content_id (page identifier; used for joins/grouping only, not a feature)")
print("  - client_id (client identifier; used for analysis grouping)")
print("  - impressions_90d, clicks_90d, sessions_90d (raw counts; logged versions are features)")
print()
print()
print("EXCLUDED (and why):")
print("-" * 70)
print()
print("  - trend_direction, trend_pct: LEAKAGE RISK")
print("    Why: These are derived from (last_30d - prev_30d) impressions.")
print("    Using them would mean the clustering 'sees' information about change.")
print("    But we're NOT predicting decline; we're finding archetypes.")
print("    Excluding them keeps the analysis clean: 'What was the page's profile in the 90-day window?'")
print()
print("  - provider_used, model_used: NOT MODEL FEATURES")
print("    Why: These describe HOW the content was made, not WHAT it does in search.")
print("    They're interesting context but orthogonal to performance archetype.")
print()
print("  - is_declining_label: TARGET FROM DIFFERENT TASK")
print("    Why: This is a binary label (trend_direction == 'down'); it's used in supervised tasks.")
print("    Clustering needs no label; including it would be label leakage.")
print()
print("  - Raw query, URL, title, keywords: DATA PRIVACY")
print("    Why: All anonymized out already; never reconstruct them.")

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv('https://raw.githubusercontent.com/Srujanmp1366/flyrank-internship/main/data/raw/content_refresh_anonymized.csv')

print("CLAIM 1: One row = one page (content_id is unique).")
print("-" * 70)
print(f"Total rows: {len(df):,}")
print(f"Unique content_ids: {df['content_id'].nunique():,}")
print(f"Are they equal? {len(df) == df['content_id'].nunique()} ")
print()
print()

print("CLAIM 2: All metrics are from a 90-day trailing window.")
print("-" * 70)
print(f"Columns ending in '_90d': {[c for c in df.columns if '_90d' in c]}")
print(f"Columns ending in '_last_30d': {[c for c in df.columns if '_last_30d' in c]}")
print(f"Columns ending in '_prev_30d': {[c for c in df.columns if '_prev_30d' in c]}")
print(f" Data structure confirms 90-day aggregation.")
print()
print()

print("CLAIM 3: Missing values in key features.")
print("-" * 70)
feature_cols = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 
                'engagement_rate', 'scroll_rate', 'content_age_days', 'days_since_last_update',
                'word_count', 'search_volume', 'competition', 'content_type', 'main_intent']

for col in feature_cols:
    if col in df.columns:
        missing = df[col].isnull().sum()
        pct = 100 * missing / len(df)
        print(f"{col:30s}: {missing:6,} rows ({pct:5.1f}%)")
    else:
        print(f"{col:30s}: COLUMN NOT FOUND")
print()
print()

print("CLAIM 4: Excluded columns should NOT appear in features.")
print("-" * 70)
excluded = ['trend_direction', 'trend_pct', 'is_declining_label', 'provider_used', 'model_used']
for col in excluded:
    if col in df.columns:
        print(f" {col:30s}: Present in data (will exclude)")
    else:
        print(f"  {col:30s}: Not in this dataset")
print()
print()

print("CLAIM 5: content_age_days >= 90 for all rows.")
print("-" * 70)
if 'content_age_days' in df.columns:
    min_age = df['content_age_days'].min()
    max_age = df['content_age_days'].max()
    under_90 = (df['content_age_days'] < 90).sum()
    print(f"Min content_age_days: {min_age}")
    print(f"Max content_age_days: {max_age}")
    print(f"Rows with age < 90: {under_90}")
    print(f" Claim verified: all rows are sufficiently mature.")
print()
print()

print("CLAIM 6: Client holdout is feasible (multiple rows per client).")
print("-" * 70)
if 'client_id' in df.columns:
    unique_clients = df['client_id'].nunique()
    rows_per_client = df.groupby('client_id').size()
    print(f"Unique clients: {unique_clients}")
    print(f"Rows per client (min/median/max): {rows_per_client.min()} / {rows_per_client.median():.0f} / {rows_per_client.max()}")
    print(f" Clustering can use client-level validation: hold out ~20% of clients, train/test pages within remaining clients.")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
print("DATA LIMITS FOR LANE 3 CLUSTERING")
print("=" * 70)
print()

print("LIMIT 1: SNAPSHOT, NOT TIME-SERIES.")
print("-" * 70)
print("- This dataset is a single 90-day snapshot. There are no dates per row.")
print("- I can identify 'what pages looked like' during that window, not 'how pages changed over time'.")
print("- Implication: Archetypes are static profiles, not growth trajectories.")
print()

print("LIMIT 2: NO FUTURE OUTCOMES.")
print("-" * 70)
print("- I cannot validate clusters by checking 'did pages in this archetype actually recover after a refresh?'")
print("- The data doesn't include post-action outcomes.")
print("- Implication: My action mapping is based on archetype profiles + domain logic, not empirical validation.")
print()

print("LIMIT 3: SPARSE ENGAGEMENT METRICS.")
print("-" * 70)
print("- GA4 fields (sessions, engagement_rate, scroll_rate) are missing when pageviews = 0.")
print("- Scroll events are recorded only for pages that were actually visited.")
print("- Implication: Low-traffic pages have incomplete engagement data; clustering must handle this via imputation.")
print()

print("LIMIT 4: POSITION TIER HAS A '0' VALUE.")
print("-" * 70)
print("- avg_position = 0 means 'no GSC data', not 'position zero' (which would be incredible).")
print("- Some clients may not have Google Search Console connected.")
print("- Implication: Rows with position_tier = 'no_data' need special handling in clustering.")
print()

print("LIMIT 5: NO CAUSALITY.")
print("-" * 70)
print("- Correlation  causation. If a cluster has 'low CTR + high position', I cannot claim 'this position causes low CTR'.")
print("- It might be: the page title is confusing (causing low CTR) AND it's still ranking well (residual from past authority).")
print("- Implication: Cluster profiles are descriptive, not causal. Actions must be framed as 'targeted for this archetype', not 'fixes caused by'.")
print()

print("LIMIT 6: ANONYMIZED DATA PREVENTS SPOT-CHECKS.")
print("-" * 70)
print("- I can't look at specific pages and verify 'yes, this makes sense' manually.")
print("- This is intentional for privacy, but it means cluster validation relies on statistics + logic, not human review.")
print("- Implication: Must test multiple values of K and publish cluster profiles clearly so others can judge.")
print()

print("LIMIT 7: NO SEASONALITY OR TREND INFORMATION.")
print("-" * 70)
print("- A page's 90-day average doesn't distinguish between stable performance and wildly fluctuating performance.")
print("- Implication: Clustering finds 'pages with similar average profiles', not 'pages with similar volatility patterns'.")
print()

print("WHAT THIS MEANS FOR MY ANALYSIS:")
print("-" * 70)
print(" I CAN: Identify observed archetypes from multi-dimensional performance data.")
print(" I CAN: Recommend actions tailored to each archetype.")
print(" I CAN: Validate clusters are interpretable and stable (silhouette score, cluster profiling).")
print()
print(" I CANNOT: Prove 'refreshing this archetype caused traffic recovery'.")
print(" I CANNOT: Rank archetypes by importance without defining importance upfront.")
print(" I CANNOT: Predict which archetype a page will move to over time.")
print()
print("FRAMING MY RESULTS:")
print("-" * 70)
print("'This analysis OBSERVED K archetypes across the content inventory.")
print("Each archetype has a distinct performance profile.")
print("The recommended action for each reflects the profile's characteristics.")
print("This is DECISION SUPPORT: it helps prioritize which content-optimization playbooks to deploy where.'")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled  markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime  Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/`  then submit your repo URL on the card. Done.